# Segmentación con SentenceSplitter — VERSIÓN CORREGIDA (v6, Pinecone + BGE-m3)
## `chunk_size=512` · `chunk_overlap=64`

Aplica `SentenceSplitter` de LlamaIndex al JSON de noticias financieras enriquecidas,
prepara la salida para **vectorizar con BGE-m3** e **indexar en Pinecone**.

**Entrada:** `noticias_enriquecido.json` — 1 documento por noticia.
**Salida:** `noticias_nodes.json` — 1 entrada por chunk, lista para Pinecone (`id` + `metadata` planos, sin `None`).

---
### Qué cambia en v6 respecto a v5

1. **Tokenizer de BGE-m3 obligatorio (no silencioso).** La carga de
   `AutoTokenizer.from_pretrained("BAAI/bge-m3")` es ahora la única fuente de
   verdad para `_count_tokens()`. Si `REQUIRE_BGE_M3_TOKENIZER = True` (default)
   y la carga falla, el notebook **lanza una excepción y detiene la ejecución**
   en vez de continuar en silencio con un tokenizer distinto (tiktoken), que
   desalinearía el `chunk_size=512` real frente a lo que BGE-m3 codifica.
   `enforce_min_overlap` usa la misma `_count_tokens()`, así que ambos están
   garantizados a coincidir exactamente.

2. **Salida compatible con Pinecone.** `node_to_dict` ahora:
   - Incluye un campo `"id"` (string, igual a `node_id`) — el nombre que
     Pinecone espera para el identificador del vector.
   - Limpia todos los metadatos heredados: Pinecone **no acepta `None`** ni
     tipos complejos en metadata. Los campos numéricos (`price_t0..t20`,
     `return_*d`) pasan `None → 0.0`; los campos de texto/categóricos
     (`market_reaction_*d`, `enrich_error`, etc.) pasan `None → ""`.
   - Añade un flag booleano `has_market_data` por nodo: como convertir `None`
     a `0.0` hace indistinguible "sin dato" de "variación 0%", este flag deja
     esa distinción disponible para filtrar en Pinecone sin perder la
     compatibilidad de tipos que exige la propia base de datos.

3. **Contrato de overlap con bloqueo real.** La verificación de overlap se
   adelanta a **antes** de serializar y escribir el JSON (ya no es una
   comprobación informativa al final). Si el porcentaje de pares con
   `overlap = 0` supera el 5%, se lanza una excepción (`OverlapValidationError`)
   y el notebook se detiene: **no llega a generarse `noticias_nodes.json`**
   si la estructura no es consistente.

4. **Inyección de identificadores macro en el texto (para BGE-m3).** El
   `text` final de cada nodo antepone `Título / Periodo / Régimen de mercado`
   antes del contenido, para reforzar la recuperación semántica multi-idioma.
   **Detalle de orden importante:** esta inyección ocurre **después** de la
   validación de overlap (punto 3), no antes. El prefijo es distinto por
   documento pero igual para todos los nodos de un mismo documento, así que
   si se inyectara antes de medir el overlap, desplazaría el punto de
   coincidencia entre nodos consecutivos y la verificación reportaría
   falsos negativos (overlap aparentemente roto cuando en realidad no lo
   está). Por eso el pipeline mantiene dos fases claramente separadas:
   validar sobre texto crudo → construir el texto final enriquecido.

## 1. Instalación

In [ ]:
!pip install llama-index-core transformers sentencepiece -q

## 2. Importaciones

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document, NodeRelationship
from google.colab import files

## 3. Carga del archivo enriquecido

Sube el archivo `noticias_enriquecido.json` generado por el notebook de enriquecimiento.

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Documentos cargados : {len(data)}")
print(f"Campos              : {list(data[0].keys())}")

## 4. Parámetros del splitter y tokenizer de BGE-m3 (obligatorio)

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `chunk_size` | 512 | Límite de la ventana efectiva de la mayoría de modelos de embedding |
| `chunk_overlap` | 64 | ~12,5 % del chunk; preserva contexto en texto financiero denso en entidades |
| `tokenizer` | tokenizer de `BAAI/bge-m3` — **obligatorio, no fallback silencioso** | El conteo de tokens debe coincidir exactamente con la codificación real de BGE-m3 |

`REQUIRE_BGE_M3_TOKENIZER = True`: si no se puede cargar el tokenizer real de
BGE-m3 (p. ej. sin conexión a Hugging Face), el notebook **se detiene aquí**
con un error explícito en vez de continuar con un tokenizer que desalinearía
el `chunk_size` configurado frente al `chunk_size` real que verá BGE-m3.
Puedes poner el flag a `False` para permitir un fallback a tiktoken si
necesitas ejecutar el notebook sin conexión — pero entonces los conteos de
`chunk_token_count` dejan de ser exactos para BGE-m3, así que no se
recomienda para la generación final del índice.

No se fija `paragraph_separator` (ver v3 en el historial de versiones):
fijarlo a `"

"` provocó overlap=0 en ~30% de los nodos consecutivos.

In [ ]:
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 64
REQUIRE_BGE_M3_TOKENIZER = True  # ver celda de arriba antes de cambiar a False

tokenizer = None
tokenizer_source = None

try:
    from transformers import AutoTokenizer
    hf_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
    tokenizer = lambda text: hf_tokenizer.encode(text, add_special_tokens=False)
    tokenizer_source = "BAAI/bge-m3 (transformers.AutoTokenizer)"
except Exception as e:
    if REQUIRE_BGE_M3_TOKENIZER:
        raise RuntimeError(
            "No se pudo cargar el tokenizer real de BAAI/bge-m3 "
            f"({type(e).__name__}: {e}). El conteo de tokens no coincidiria "
            "con BGE-m3 y el chunk_size=512 dejaria de ser exacto. "
            "Verifica la conexion a internet/Hugging Face, o pon "
            "REQUIRE_BGE_M3_TOKENIZER = False si aceptas un fallback aproximado."
        ) from e
    print(f"[aviso] No se pudo cargar el tokenizer de BGE-m3 ({e}).")
    print("        Fallback a tiktoken (aproximado, NO exacto para BGE-m3).")
    tokenizer_source = "fallback: default (tiktoken cl100k_base)"

splitter_kwargs = dict(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    # NO fijamos paragraph_separator: ver celda de introduccion (v3 causo
    # overlap=0 en ~30% de los nodos al tratar parrafos como unidades atomicas).
)
if tokenizer is not None:
    splitter_kwargs["tokenizer"] = tokenizer

splitter = SentenceSplitter(**splitter_kwargs)


def _count_tokens(text: str) -> int:
    if tokenizer is not None:
        return len(tokenizer(text))
    return len(splitter._tokenizer(text))  # solo si REQUIRE_BGE_M3_TOKENIZER=False


print(f"SentenceSplitter configurado: chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}")
print(f"Tokenizer usado para medir tokens: {tokenizer_source}")

## 5. Construcción de objetos `Document`

Cada documento de LlamaIndex lleva el `contenido` como texto principal
y los demás campos como metadatos — serán heredados por todos sus nodos.

Se marcan todas las claves de metadata como `excluded_embed_metadata_keys` /
`excluded_llm_metadata_keys`: siguen disponibles en `node.metadata` (y en el
JSON final) pero dejan de restarse del presupuesto de tokens de cada chunk.

In [ ]:
documents = []
for doc in data:
    if not doc.get("contenido"):
        continue
    meta = {k: v for k, v in doc.items() if k != "contenido"}
    d = Document(text=doc["contenido"], metadata=meta)
    d.excluded_embed_metadata_keys = list(meta.keys())
    d.excluded_llm_metadata_keys = list(meta.keys())
    documents.append(d)

print(f"Documentos listos para segmentar: {len(documents)}")

## 6. Segmentación

`get_nodes_from_documents` respeta límites de oración: nunca parte una frase por la mitad.
Cada nodo hereda los metadatos del documento padre y almacena referencias
al nodo anterior, siguiente y al documento fuente.

In [ ]:
nodes = splitter.get_nodes_from_documents(documents, show_progress=True)

print(f"\nDocumentos originales : {len(documents)}")
print(f"Nodos generados       : {len(nodes)}")
print(f"Media nodos/documento : {len(nodes) / len(documents):.1f}")

## 6bis. Reparación explícita de overlap

`SentenceSplitter` trocea por unidades completas (oraciones) y nunca las
subdivide para ajustar el overlap: si la última oración de un chunk ya
supera por sí sola el presupuesto de `chunk_overlap`, se descarta entera y
el nodo siguiente queda sin contexto compartido.

Esta celda repara esos casos **después** de generar los nodos: si el overlap
real entre un nodo y el siguiente es insuficiente, antepone manualmente el
final del nodo anterior al inicio del nodo siguiente. El tamaño del
fragmento a inyectar se acota por **tokens reales** (medidos con la misma
`_count_tokens` que usa BGE-m3), con un tope de 1.5× `chunk_overlap` — esto
es deliberado: contar por número de palabras es poco fiable en texto
financiero denso en números/tickers, y se rompe por completo con bloques de
texto corrupto sin espacios (p. ej. widgets de cotizaciones mal scrapeados),
que de otro modo se tratarían como una única "palabra" gigante e inflarían
el chunk sin control. Nunca mezcla contenido entre documentos distintos.

In [ ]:
def _longest_common_overlap(a_tail: str, b_head: str) -> int:
    max_k = min(len(a_tail), len(b_head))
    for k in range(max_k, 0, -1):
        if a_tail[-k:] == b_head[:k]:
            return k
    return 0


def _tail_by_token_budget(text: str, overlap_tokens: int, count_tokens_fn, max_ratio: float = 1.5) -> str:
    '''Devuelve un sufijo de text cuyo tamano en TOKENS REALES se aproxima a
    overlap_tokens, sin superar overlap_tokens * max_ratio. Camina hacia atras
    palabra a palabra (preservando el formato original exacto) y mide el
    conteo de tokens real en cada paso.'''
    tokens_ws = re.split(r"(\s+)", text)
    word_positions = [i for i, t in enumerate(tokens_ws) if t.strip() != ""]
    if not word_positions:
        return text

    max_cap = overlap_tokens * max_ratio
    best = ""
    for k in range(1, len(word_positions) + 1):
        start_idx = word_positions[-k]
        candidate = "".join(tokens_ws[start_idx:])
        candidate_tokens = count_tokens_fn(candidate)
        if candidate_tokens > max_cap:
            break
        best = candidate
        if candidate_tokens >= overlap_tokens:
            break

    if best:
        return best

    tail_chars = int(max_cap * 4)  # aproximacion conservadora tokens -> caracteres
    return text[-tail_chars:]


def enforce_min_overlap(nodes, overlap_tokens: int, min_overlap_chars: int = 20):
    '''Garantiza overlap real entre nodos consecutivos del mismo documento.
    Nunca mezcla overlap entre documentos distintos (usa NodeRelationship.SOURCE).'''
    node_map = {n.node_id: n for n in nodes}
    fixed = 0

    for n in nodes:
        next_rel = n.relationships.get(NodeRelationship.NEXT)
        if not next_rel:
            continue
        nxt = node_map.get(next_rel.node_id)
        if nxt is None:
            continue

        src_n   = n.relationships.get(NodeRelationship.SOURCE)
        src_nxt = nxt.relationships.get(NodeRelationship.SOURCE)
        if not src_n or not src_nxt or src_n.node_id != src_nxt.node_id:
            continue  # nunca mezclar overlap entre documentos distintos

        overlap_now = _longest_common_overlap(n.text[-500:], nxt.text[:500])
        if overlap_now >= min_overlap_chars:
            continue  # ya hay overlap util, no se toca

        tail = _tail_by_token_budget(n.text, overlap_tokens, _count_tokens)
        sep = "" if tail.endswith((" ", "\n", "\t")) or nxt.text.startswith((" ", "\n", "\t")) else " "
        nxt.text = tail + sep + nxt.text
        fixed += 1

    print(f"Overlap reparado manualmente en {fixed} de {len(nodes)} nodos "
          f"({fixed/len(nodes)*100:.1f}%).")
    return nodes


nodes = enforce_min_overlap(nodes, overlap_tokens=CHUNK_OVERLAP)

## 7. Validación obligatoria del contrato de overlap (bloqueante)

**Esta celda debe ejecutarse ANTES de construir el JSON final**, y sobre el
texto crudo de los nodos (antes de inyectar el prefijo de metadatos macro en
el paso 8) — si se hiciera después, el prefijo desplazaría el punto de
coincidencia entre nodos y la comprobación daría falsos negativos.

Si el porcentaje de pares consecutivos con `overlap = 0` supera el
**5%** (umbral configurable en `MAX_ZERO_OVERLAP_RATIO`), se lanza
`OverlapValidationError` y el notebook se detiene aquí: no se llega a
generar `noticias_nodes.json`. Esto convierte la comprobación de
"informativa" a "bloqueante", tal y como exige el contrato de calidad de
overlap del pipeline.

In [ ]:
import statistics

MAX_ZERO_OVERLAP_RATIO = 0.05  # 5%


class OverlapValidationError(Exception):
    pass


def validate_overlap_contract(nodes, max_zero_ratio: float = MAX_ZERO_OVERLAP_RATIO):
    node_map = {n.node_id: n for n in nodes}
    overlap_lengths = []

    for n in nodes:
        next_rel = n.relationships.get(NodeRelationship.NEXT)
        if not next_rel:
            continue
        nxt = node_map.get(next_rel.node_id)
        if nxt is None:
            continue
        src_n   = n.relationships.get(NodeRelationship.SOURCE)
        src_nxt = nxt.relationships.get(NodeRelationship.SOURCE)
        if not src_n or not src_nxt or src_n.node_id != src_nxt.node_id:
            continue

        k = _longest_common_overlap(n.text[-700:], nxt.text[:700])
        overlap_lengths.append(k)

    if not overlap_lengths:
        print("[aviso] No hay pares de nodos consecutivos que validar (corpus de un solo nodo por documento?).")
        return

    zero = sum(1 for x in overlap_lengths if x == 0)
    ratio = zero / len(overlap_lengths)

    print(f"Pares de nodos consecutivos analizados : {len(overlap_lengths)}")
    print(f"Overlap medio (caracteres)              : {statistics.mean(overlap_lengths):.1f}")
    print(f"Pares con overlap = 0                   : {zero} ({ratio*100:.2f}%)")
    print(f"Umbral maximo permitido                 : {max_zero_ratio*100:.1f}%")

    if ratio > max_zero_ratio:
        raise OverlapValidationError(
            f"El {ratio*100:.2f}% de los nodos consecutivos no tienen overlap real, "
            f"por encima del umbral permitido ({max_zero_ratio*100:.1f}%). "
            "Se detiene la ejecucion: revisa enforce_min_overlap antes de generar el JSON final."
        )

    print("OK: el overlap cumple el contrato de calidad. Se puede continuar con la serializacion.")


validate_overlap_contract(nodes)

## 8. Serialización a JSON compatible con Pinecone

`node_to_dict` construye, por nodo:

- `"id"`: string único (igual a `node_id`) — campo que Pinecone espera para el identificador del vector.
- `"text"`: el chunk final, con un prefijo de identificadores macro
  (Título / Periodo / Régimen de mercado) antepuesto para reforzar la
  recuperación semántica multi-idioma en BGE-m3. `chunk_token_count` se
  recalcula sobre este texto final (el que realmente se va a vectorizar),
  no sobre el texto crudo del splitter.
- **Metadatos limpios para Pinecone** (sin `None`, sin tipos complejos):
  - Numéricos (`price_t0..t20`, `return_1d/5d/20d`) → `None` se convierte en `0.0`.
  - Texto/categóricos (`market_reaction_*d`, `enrich_error`, etc.) → `None` se convierte en `""`.
  - Se añade `has_market_data` (booleano): como `0.0` no distingue "sin dato"
    de "variación real del 0%", este flag preserva esa información para
    poder filtrar en Pinecone sin violar las restricciones de tipos de metadata.

In [ ]:
NUMERIC_METADATA_FIELDS = [
    "price_t0", "price_t1", "price_t5", "price_t20",
    "return_1d", "return_5d", "return_20d",
]
STRING_METADATA_FIELDS = [
    "titulo", "url", "fecha", "periodo", "language", "indice_sector", "regimen_mercado",
    "ticker", "market_reaction_1d", "market_reaction_5d", "market_reaction_20d",
    "enrich_status", "enrich_error",
]


def _clean_numeric(value):
    if value is None:
        return 0.0
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0


def _clean_string(value):
    if value is None:
        return ""
    return str(value)


MACRO_PREFIX_TEMPLATE = "Titulo: {titulo}\nPeriodo: {periodo} | Regimen de mercado: {regimen_mercado}\n\n"


def node_to_dict(node) -> dict:
    '''Convierte un TextNode de LlamaIndex en un diccionario serializable y
    compatible con Pinecone (id explicito, metadata sin None).'''
    prev_id   = node.relationships.get(NodeRelationship.PREVIOUS, None)
    next_id   = node.relationships.get(NodeRelationship.NEXT,     None)
    source_id = node.relationships.get(NodeRelationship.SOURCE,   None)

    meta = node.metadata
    titulo          = _clean_string(meta.get("titulo"))
    periodo         = _clean_string(meta.get("periodo"))
    regimen_mercado = _clean_string(meta.get("regimen_mercado"))

    prefix = MACRO_PREFIX_TEMPLATE.format(
        titulo=titulo or "N/D",
        periodo=periodo or "N/D",
        regimen_mercado=regimen_mercado or "N/D",
    )
    final_text = prefix + node.text

    has_market_data = any(meta.get(f) is not None for f in NUMERIC_METADATA_FIELDS)

    row = {
        "id"             : node.node_id,
        "node_id"        : node.node_id,
        "source_doc_id"  : source_id.node_id  if source_id  else "",
        "prev_node_id"   : prev_id.node_id    if prev_id    else "",
        "next_node_id"   : next_id.node_id    if next_id    else "",
        "chunk_size"       : CHUNK_SIZE,
        "chunk_overlap"    : CHUNK_OVERLAP,
        "chunk_token_count": _count_tokens(final_text),
        "text"           : final_text,
        "has_market_data": has_market_data,
        # ── metadatos heredados del documento padre (limpios para Pinecone) ──
        "titulo"         : titulo,
        "url"            : _clean_string(meta.get("url")),
        "fecha"          : _clean_string(meta.get("fecha")),
        "periodo"        : periodo,
        "language"       : _clean_string(meta.get("language")),
        "indice_sector"  : _clean_string(meta.get("indice_sector")),
        "regimen_mercado": regimen_mercado,
        "ticker"            : _clean_string(meta.get("ticker")),
        "price_t0"          : _clean_numeric(meta.get("price_t0")),
        "price_t1"          : _clean_numeric(meta.get("price_t1")),
        "price_t5"          : _clean_numeric(meta.get("price_t5")),
        "price_t20"         : _clean_numeric(meta.get("price_t20")),
        "return_1d"         : _clean_numeric(meta.get("return_1d")),
        "return_5d"         : _clean_numeric(meta.get("return_5d")),
        "return_20d"        : _clean_numeric(meta.get("return_20d")),
        "market_reaction_1d" : _clean_string(meta.get("market_reaction_1d")),
        "market_reaction_5d" : _clean_string(meta.get("market_reaction_5d")),
        "market_reaction_20d": _clean_string(meta.get("market_reaction_20d")),
        "enrich_status"     : _clean_string(meta.get("enrich_status")),
        "enrich_error"      : _clean_string(meta.get("enrich_error")),
    }
    return row


# Si llegamos aqui es porque validate_overlap_contract() paso sin excepcion.
output = [node_to_dict(n) for n in nodes]

# Verificacion de compatibilidad con Pinecone: ningun valor None en metadata
_none_fields = set()
for row in output:
    for k, v in row.items():
        if v is None:
            _none_fields.add(k)
assert not _none_fields, f"Quedan campos con None, Pinecone los rechazaria: {_none_fields}"

OUTPUT_FILE = "noticias_nodes.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Guardado: {OUTPUT_FILE}  ({len(output)} nodos, {Path(OUTPUT_FILE).stat().st_size/1024/1024:.1f} MB)")
print("Verificacion Pinecone: sin valores None en metadata. OK.")

## 9. Estadísticas del corpus segmentado

In [ ]:
toks = [n["chunk_token_count"] for n in output]
print("Tokens por nodo (texto final, con prefijo de metadatos macro):")
print(f"  media   : {statistics.mean(toks):.1f}")
print(f"  mediana : {statistics.median(toks):.1f}")
print(f"  max     : {max(toks)}")
print(f"  min     : {min(toks)}")

print("\nNodos por índice / sector:")
for sector, cnt in sorted(Counter(n["indice_sector"] for n in output).items(), key=lambda x: -x[1]):
    pct = cnt / len(output) * 100
    print(f"  {sector:<35} {cnt:>5}  ({pct:.1f}%)")

print("\nNodos por régimen de mercado:")
for reg, cnt in Counter(n["regimen_mercado"] for n in output).items():
    pct = cnt / len(output) * 100
    print(f"  {reg:<12} {cnt:>5}  ({pct:.1f}%)")

print("\nNodos por idioma:")
for lang, cnt in Counter(n["language"] for n in output).items():
    pct = cnt / len(output) * 100
    print(f"  {lang:<8} {cnt:>5}  ({pct:.1f}%)")

sin_market_data = sum(1 for n in output if not n["has_market_data"])
print(f"\nNodos sin datos de mercado (has_market_data=False): {sin_market_data} ({sin_market_data/len(output)*100:.1f}%)")

## 10. Inspección visual del texto final (con prefijo de metadatos)

Muestra 2 nodos ya con el prefijo `Título / Periodo / Régimen de mercado`
aplicado, tal y como quedarán almacenados y se van a vectorizar.

In [ ]:
for row in output[:2]:
    print("=" * 80)
    print(f"id: {row['id']}")
    print(f"chunk_token_count: {row['chunk_token_count']}")
    print("-" * 80)
    print(row["text"][:400])
    print()

## 11. Descarga del archivo de nodos

In [ ]:
files.download(OUTPUT_FILE)
print(f"Descargando {OUTPUT_FILE}…")